# Proyecto final – Curso 1: Fundamentos estadísticos para ML
## Predicción de precios de viviendas (Kaggle: *House Prices – Advanced Regression Techniques*)

Este notebook recorre, celda por celda, todo lo visto en el curso: estadística descriptiva, distribuciones, inferencia, correlación, regresión lineal y, al final, un modelo de Machine Learning que predice el precio de una casa.

**Cómo usarlo en Google Colab**
1. Ejecuta las celdas **una por una, en orden**, con `Shift + Enter`.
2. Lee el texto antes de cada celda: explica qué vas a ver y qué módulo del curso aplica.
3. No hace falta GPU: el entorno por defecto de Colab es suficiente.

## 0. Preparación del entorno
Importamos las librerías. Todas vienen preinstaladas en Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")
print("Librerías cargadas ✔")

## 1. Carga de datos
Hay tres formas de obtener `train.csv` y `test.csv`. Cambia la variable `METODO`:

- `"mirror"` (por defecto): descarga una copia pública del dataset alojada en GitHub. Es lo más rápido para practicar.
- `"kaggle"`: usa la API oficial de Kaggle. Necesitas tu `kaggle.json` (Kaggle → Settings → *Create New Token*) y haber aceptado las reglas de la competencia en su página.
- `"upload"`: sube tú mismo los archivos descargados desde Kaggle.

In [ ]:
METODO = "mirror"   # "mirror" | "kaggle" | "upload"

if METODO == "mirror":
    base = "https://raw.githubusercontent.com/hjhuney/Data/master/AmesHousing/"
    train = pd.read_csv(base + "train.csv")
    test = pd.read_csv(base + "test.csv")

elif METODO == "kaggle":
    from google.colab import files
    print("Sube tu kaggle.json")
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c house-prices-advanced-regression-techniques
    !unzip -o -q house-prices-advanced-regression-techniques.zip
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

elif METODO == "upload":
    from google.colab import files
    print("Sube train.csv y test.csv")
    files.upload()
    train = pd.read_csv("train.csv")
    test = pd.read_csv("test.csv")

print("train:", train.shape, "| test:", test.shape)

## 2. Primer vistazo a los datos  *(Módulo 1 – Estadística descriptiva)*
Cada fila es una casa de Ames (Iowa) y cada columna una característica. La variable que queremos predecir es **`SalePrice`**.

In [ ]:
train.head()

In [ ]:
numericas = train.select_dtypes(include="number").columns
categoricas = train.select_dtypes(exclude="number").columns
print(f"Variables numéricas: {len(numericas)}")
print(f"Variables categóricas: {len(categoricas)}")

Calculamos a mano las medidas de tendencia central y dispersión de `SalePrice`, y detectamos outliers con la regla del IQR (valores fuera de Q1 − 1.5·IQR y Q3 + 1.5·IQR).

In [ ]:
precio = train["SalePrice"]

q1, q3 = precio.quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Media:     ${precio.mean():,.0f}")
print(f"Mediana:   ${precio.median():,.0f}")
print(f"Moda:      ${precio.mode()[0]:,.0f}")
print(f"Desv. std: ${precio.std():,.0f}")
print(f"Q1 / Q3:   ${q1:,.0f} / ${q3:,.0f}   (IQR = ${iqr:,.0f})")
print(f"Outliers por IQR: {((precio < lim_inf) | (precio > lim_sup)).sum()} casas")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(precio, kde=True, ax=ax[0])
ax[0].axvline(precio.mean(), color="red", ls="--", label="media")
ax[0].axvline(precio.median(), color="green", ls="--", label="mediana")
ax[0].legend(); ax[0].set_title("Distribución de SalePrice")
sns.boxplot(x=precio, ax=ax[1]); ax[1].set_title("Boxplot de SalePrice")
plt.show()

**Observa:** la media es mayor que la mediana y hay una cola larga a la derecha. Eso indica una distribución **asimétrica positiva**.

## 3. Distribuciones y transformaciones  *(Módulo 3)*
La regresión lineal funciona mejor cuando los errores se distribuyen de forma aproximadamente normal. Medimos la asimetría (*skewness*) y la curtosis, y probamos la transformación logarítmica.

In [ ]:
log_precio = np.log1p(precio)

print(f"Skewness original: {stats.skew(precio):.2f}   | Curtosis: {stats.kurtosis(precio):.2f}")
print(f"Skewness log:      {stats.skew(log_precio):.2f}   | Curtosis: {stats.kurtosis(log_precio):.2f}")

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
sns.histplot(precio, kde=True, ax=ax[0, 0]); ax[0, 0].set_title("SalePrice")
stats.probplot(precio, plot=ax[0, 1]); ax[0, 1].set_title("Q-Q plot SalePrice")
sns.histplot(log_precio, kde=True, ax=ax[1, 0], color="purple"); ax[1, 0].set_title("log(SalePrice)")
stats.probplot(log_precio, plot=ax[1, 1]); ax[1, 1].set_title("Q-Q plot log(SalePrice)")
plt.tight_layout(); plt.show()

En el Q-Q plot, cuanto más se pegan los puntos a la línea roja, más normal es la distribución. Después del logaritmo la distribución se parece mucho más a una normal. **Por eso el modelo predecirá `log(SalePrice)` y al final volveremos a dólares con la función inversa.**

## 4. Inferencia estadística  *(Módulo 4)*
### 4.1 Intervalo de confianza del precio medio
No conocemos el precio medio de *todas* las casas de Ames, solo el de esta muestra. El intervalo de confianza al 95% nos da un rango plausible para la media poblacional.

In [ ]:
n = len(precio)
media = precio.mean()
error_std = precio.std() / np.sqrt(n)
ic = stats.t.interval(0.95, df=n - 1, loc=media, scale=error_std)
print(f"Media muestral: ${media:,.0f}")
print(f"IC 95%: ${ic[0]:,.0f} – ${ic[1]:,.0f}")

### 4.2 Prueba de hipótesis: ¿el aire acondicionado central cambia el precio?
- **H₀:** el precio medio es igual con y sin aire acondicionado central.
- **H₁:** los precios medios son distintos.

Usamos una prueba t de Welch (no asume varianzas iguales).

In [ ]:
con_ac = train.loc[train["CentralAir"] == "Y", "SalePrice"]
sin_ac = train.loc[train["CentralAir"] == "N", "SalePrice"]

t, p = stats.ttest_ind(con_ac, sin_ac, equal_var=False)
print(f"Con A/C: n={len(con_ac)}, media=${con_ac.mean():,.0f}")
print(f"Sin A/C: n={len(sin_ac)}, media=${sin_ac.mean():,.0f}")
print(f"t = {t:.2f}, p-valor = {p:.2e}")
print("Rechazamos H0 (diferencia significativa)" if p < 0.05 else "No rechazamos H0")

### 4.3 ANOVA: ¿la calidad general cambia el precio?
Con más de dos grupos usamos ANOVA. Aquí comparamos el precio entre los niveles de `OverallQual` (1 a 10).

In [ ]:
grupos = [g["SalePrice"].values for _, g in train.groupby("OverallQual")]
f, p = stats.f_oneway(*grupos)
print(f"F = {f:.1f}, p-valor = {p:.2e}")

plt.figure(figsize=(10, 4))
sns.boxplot(data=train, x="OverallQual", y="SalePrice")
plt.title("SalePrice según OverallQual"); plt.show()

## 5. Correlación  *(Módulo 5)*
¿Qué variables numéricas se mueven junto con el precio?

In [ ]:
corr = train[numericas].corr()["SalePrice"].drop("SalePrice").sort_values(ascending=False)
print("Top 10 correlaciones con SalePrice:")
print(corr.head(10).round(3))

top = corr.head(8).index.tolist() + ["SalePrice"]
plt.figure(figsize=(9, 7))
sns.heatmap(train[top].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matriz de correlación (top variables)"); plt.show()

**Pearson vs. Spearman:** Pearson mide relación *lineal*; Spearman mide relación *monótona* usando rangos y es menos sensible a outliers.

In [ ]:
pear = stats.pearsonr(train["GrLivArea"], train["SalePrice"])[0]
spear = stats.spearmanr(train["GrLivArea"], train["SalePrice"])[0]
print(f"GrLivArea vs SalePrice → Pearson: {pear:.3f} | Spearman: {spear:.3f}")

sospechosos = train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)]
plt.figure(figsize=(8, 5))
plt.scatter(train["GrLivArea"], train["SalePrice"], alpha=0.4)
plt.scatter(sospechosos["GrLivArea"], sospechosos["SalePrice"], color="red", s=80, label="outliers")
plt.xlabel("GrLivArea (pies² habitables)"); plt.ylabel("SalePrice"); plt.legend()
plt.title("Área habitable vs precio"); plt.show()
print(f"Casas enormes y baratas (outliers): {len(sospechosos)}")

Esas dos casas en rojo son muy grandes pero baratísimas: rompen la tendencia. El propio autor del dataset recomienda quitarlas, y lo haremos en la limpieza.

## 6. Regresión lineal simple y múltiple  *(Módulo 5)*
### 6.1 Simple: precio en función del área
Usamos `statsmodels` porque muestra la tabla estadística completa (coeficientes, p-valores, R²).

In [ ]:
X_simple = sm.add_constant(train["GrLivArea"])
modelo_simple = sm.OLS(train["SalePrice"], X_simple).fit()
print(modelo_simple.summary().tables[1])
print(f"\nR² = {modelo_simple.rsquared:.3f}")
pendiente = modelo_simple.params["GrLivArea"]
print(f"Interpretación: cada pie² adicional se asocia con +${pendiente:,.0f} en el precio.")

### 6.2 Múltiple: agregamos calidad, garaje y sótano

In [ ]:
vars_mult = ["GrLivArea", "OverallQual", "GarageCars", "TotalBsmtSF", "YearBuilt"]
X_mult = sm.add_constant(train[vars_mult])
modelo_mult = sm.OLS(np.log1p(train["SalePrice"]), X_mult).fit()
print(modelo_mult.summary().tables[1])
print(f"\nR² = {modelo_mult.rsquared:.3f}  (con log(SalePrice) como objetivo)")

Como el objetivo está en logaritmo, un coeficiente β se interpreta aproximadamente como **un cambio de 100·β % en el precio** por unidad de la variable. Por ejemplo, si el coeficiente de `OverallQual` es 0.12, subir un punto de calidad se asocia con ~12% más de precio.

### 6.3 Diagnóstico de supuestos
Revisamos los residuos (deben verse como ruido alrededor de 0, sin forma) y la multicolinealidad con VIF (valores > 5–10 son preocupantes).

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].scatter(modelo_mult.fittedvalues, modelo_mult.resid, alpha=0.4)
ax[0].axhline(0, color="red"); ax[0].set_title("Residuos vs valores ajustados")
ax[0].set_xlabel("Predicción (log)"); ax[0].set_ylabel("Residuo")
stats.probplot(modelo_mult.resid, plot=ax[1]); ax[1].set_title("Q-Q plot de residuos")
plt.tight_layout(); plt.show()

vif = pd.Series(
    [variance_inflation_factor(X_mult.values, i) for i in range(1, X_mult.shape[1])],
    index=vars_mult, name="VIF")
print(vif.round(2))

## 7. Limpieza de datos  *(Proyecto – paso 2)*
Ahora preparamos **todas** las variables para el modelo de ML. Clave: en este dataset muchos `NaN` no son datos perdidos sino "no tiene" (sin piscina, sin garaje, sin sótano…).

In [ ]:
faltantes = train.isnull().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("Columnas con valores faltantes (% en train):")
print((faltantes / len(train) * 100).round(1))

Definimos la limpieza como una **función**, así podremos aplicarla igual al set de entrenamiento, al de test y a cualquier casa nueva que inventemos. Los valores de relleno (medianas, modas) se aprenden **solo del train**, para no filtrar información del test.

In [ ]:
# Quitamos los 2 outliers identificados en la sección 5
train = train.drop(train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index).reset_index(drop=True)
y = np.log1p(train["SalePrice"])          # objetivo en escala log
train_X_raw = train.drop(columns=["SalePrice"])

# NaN que significan "no tiene"
cat_ninguno = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
               "GarageType", "GarageFinish", "GarageQual", "GarageCond",
               "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
               "MasVnrType"]
num_cero = ["GarageYrBlt", "GarageArea", "GarageCars", "BsmtFinSF1", "BsmtFinSF2",
            "BsmtUnfSF", "TotalBsmtSF", "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"]

# Valores aprendidos del train
mediana_frente_barrio = train_X_raw.groupby("Neighborhood")["LotFrontage"].median()
mediana_frente = train_X_raw["LotFrontage"].median()
modas = train_X_raw.select_dtypes(exclude="number").mode().iloc[0]
medianas = train_X_raw.select_dtypes(include="number").median()

def limpiar(df):
    df = df.copy()
    for c in cat_ninguno:
        df[c] = df[c].fillna("Ninguno")
    for c in num_cero:
        df[c] = df[c].fillna(0)
    df["LotFrontage"] = df["LotFrontage"].fillna(df["Neighborhood"].map(mediana_frente_barrio)).fillna(mediana_frente)
    for c in df.select_dtypes(exclude="number").columns:
        df[c] = df[c].fillna(modas.get(c, "Ninguno"))
    for c in df.select_dtypes(include="number").columns:
        df[c] = df[c].fillna(medianas.get(c, 0))
    return df

train_limpio = limpiar(train_X_raw)
print("Nulos restantes en train:", train_limpio.isnull().sum().sum())
print("Nulos restantes en test: ", limpiar(test).isnull().sum().sum())

## 8. Feature engineering  *(Proyecto – paso 3)*
Creamos variables nuevas con sentido de negocio, corregimos la asimetría de las numéricas más sesgadas con `log1p` y convertimos las categóricas en columnas 0/1 (*one-hot encoding*).

In [ ]:
def crear_features(df):
    df = df.copy()
    df["MSSubClass"] = df["MSSubClass"].astype(str)      # es un código, no un número
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"]
                       + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TienePiscina"] = (df["PoolArea"] > 0).astype(int)
    df["TieneGaraje"] = (df["GarageArea"] > 0).astype(int)
    return df.drop(columns=["Id"])

base = crear_features(train_limpio)

# Columnas numéricas muy asimétricas (decidido con el train)
num_cols = base.select_dtypes(include="number").columns
asimetria = base[num_cols].apply(stats.skew).sort_values(ascending=False)
cols_sesgadas = asimetria[asimetria.abs() > 0.75].index.tolist()
print(f"{len(cols_sesgadas)} columnas se transformarán con log1p. Las más sesgadas:")
print(asimetria.head(8).round(2))

In [ ]:
def preparar(df_crudo, columnas_modelo=None):
    df = crear_features(limpiar(df_crudo))
    for c in cols_sesgadas:
        df[c] = np.log1p(df[c].clip(lower=0))
    df = pd.get_dummies(df, dtype=float)
    if columnas_modelo is not None:
        df = df.reindex(columns=columnas_modelo, fill_value=0.0)
    return df

X = preparar(train_X_raw)
columnas_modelo = X.columns
X_test = preparar(test, columnas_modelo)
print("Matriz de entrenamiento:", X.shape)
print("Matriz de test:         ", X_test.shape)

## 9. Modelado  *(Módulo 6 + Proyecto – paso 4)*
Comparamos tres modelos con **validación cruzada de 5 pliegues** y la métrica oficial de Kaggle: **RMSE sobre el logaritmo del precio**.

- **Regresión lineal:** sin regularización. Con ~300 columnas puede sobreajustar.
- **Ridge (L2):** encoge todos los coeficientes.
- **Lasso (L1):** encoge y además lleva varios coeficientes exactamente a 0 (selección de variables).

`RobustScaler` estandariza las variables usando mediana e IQR, así los outliers pesan menos.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def rmse_cv(modelo):
    return np.sqrt(-cross_val_score(modelo, X, y, scoring="neg_mean_squared_error", cv=kf))

modelos = {
    "Regresión lineal": make_pipeline(RobustScaler(), LinearRegression()),
    "Ridge": make_pipeline(RobustScaler(), RidgeCV(alphas=np.logspace(-1, 3, 50))),
    "Lasso": make_pipeline(RobustScaler(), LassoCV(alphas=np.logspace(-5, -1, 50), max_iter=50000, cv=5)),
}

resultados = {}
for nombre, modelo in modelos.items():
    puntajes = rmse_cv(modelo)
    resultados[nombre] = puntajes
    print(f"{nombre:18s} RMSE(log) = {puntajes.mean():.4f} ± {puntajes.std():.4f}")

Un RMSE de ~0.11 en escala log significa que el modelo se equivoca, en promedio, alrededor de un **11%** del precio. Si la regresión lineal sale con un error mucho mayor o muy inestable, es el **overfitting** del Módulo 6 en acción; la regularización lo corrige.

In [ ]:
mejor_nombre = min(resultados, key=lambda k: resultados[k].mean())
mejor = modelos[mejor_nombre]
print("Mejor modelo:", mejor_nombre)

# Entrenamos el mejor modelo con todo el train
mejor.fit(X, y)
alpha = mejor[-1].alpha_
print(f"Alpha elegido por validación cruzada: {alpha:.5f}")
if mejor_nombre == "Lasso":
    usadas = (mejor[-1].coef_ != 0).sum()
    print(f"Lasso usa {usadas} de {X.shape[1]} variables (el resto quedó en 0)")

## 10. Evaluación en datos no vistos  *(Proyecto – paso 5)*
Separamos un 20% del train como *holdout*, entrenamos con el 80% y medimos en dólares reales.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
modelo_eval = modelos[mejor_nombre]
modelo_eval.fit(X_tr, y_tr)

pred_log = modelo_eval.predict(X_val)
real = np.expm1(y_val)
pred = np.expm1(pred_log)

print(f"RMSE (log):  {np.sqrt(mean_squared_error(y_val, pred_log)):.4f}")
print(f"MAE:         ${mean_absolute_error(real, pred):,.0f}")
print(f"R²:          {r2_score(real, pred):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].scatter(real, pred, alpha=0.5)
lim = [real.min(), real.max()]
ax[0].plot(lim, lim, color="red", ls="--")
ax[0].set_xlabel("Precio real"); ax[0].set_ylabel("Precio predicho")
ax[0].set_title("Predicho vs real")
ax[1].scatter(pred_log, y_val - pred_log, alpha=0.5)
ax[1].axhline(0, color="red")
ax[1].set_xlabel("Predicción (log)"); ax[1].set_ylabel("Residuo")
ax[1].set_title("Residuos")
plt.tight_layout(); plt.show()

# Volvemos a entrenar con todo el train para lo que sigue
mejor.fit(X, y)

## 11. ¿Cómo predice el modelo una casa?  🔍
Un modelo lineal es solo una suma: **predicción = intercepto + Σ (coeficiente × valor escalado de cada variable)**. Tomamos una casa concreta y vemos esa suma pieza por pieza.

In [ ]:
idx = 10   # cambia este número para ver otra casa
casa = X.iloc[[idx]]

escalador, regresor = mejor[0], mejor[-1]
valores_esc = escalador.transform(casa)[0]
aportes = pd.Series(regresor.coef_ * valores_esc, index=X.columns)

pred_log = regresor.intercept_ + aportes.sum()
print("Casa #", idx)
print(train_X_raw.loc[idx, ["Neighborhood", "OverallQual", "GrLivArea", "YearBuilt", "GarageCars"]].to_string())
print(f"\nIntercepto (log):              {regresor.intercept_:.4f}")
print(f"Suma de aportes (log):         {aportes.sum():+.4f}")
print(f"Predicción (log):              {pred_log:.4f}")
print(f"Predicción en dólares:         ${np.expm1(pred_log):,.0f}")
print(f"Precio real:                   ${np.expm1(y.iloc[idx]):,.0f}")

In [ ]:
top_aportes = aportes[aportes != 0].sort_values(key=abs, ascending=False).head(12)
plt.figure(figsize=(9, 5))
colores = ["seagreen" if v > 0 else "indianred" for v in top_aportes]
plt.barh(top_aportes.index[::-1], top_aportes.values[::-1], color=colores[::-1])
plt.axvline(0, color="black", lw=0.8)
plt.title(f"Variables que más empujan el precio de la casa #{idx}\n(verde = sube, rojo = baja)")
plt.xlabel("Aporte a log(precio)"); plt.show()

Como el objetivo está en logaritmo, un aporte de +0.10 equivale aproximadamente a **+10% en el precio**.

## 12. Simulador: modifica una casa y mira cómo cambia el precio
Tomamos la casa original (datos crudos), cambiamos algunas características y la pasamos por **exactamente el mismo proceso** de limpieza, features y modelo.

In [ ]:
casa_original = train_X_raw.iloc[[idx]].copy()
casa_mejorada = casa_original.copy()

# ✏️ Edita estos cambios y vuelve a ejecutar
casa_mejorada["OverallQual"] = min(10, int(casa_mejorada["OverallQual"].iloc[0]) + 2)
casa_mejorada["GrLivArea"] = casa_mejorada["GrLivArea"] + 300
casa_mejorada["2ndFlrSF"] = casa_mejorada["2ndFlrSF"] + 300   # el área extra va al 2do piso
casa_mejorada["GarageCars"] = 3

p_orig = np.expm1(mejor.predict(preparar(casa_original, columnas_modelo)))[0]
p_mej = np.expm1(mejor.predict(preparar(casa_mejorada, columnas_modelo)))[0]
print(f"Precio estimado original:  ${p_orig:,.0f}")
print(f"Precio estimado mejorado:  ${p_mej:,.0f}")
print(f"Diferencia:                ${p_mej - p_orig:+,.0f}  ({(p_mej / p_orig - 1) * 100:+.1f}%)")

## 13. Predicción sobre el test y archivo para Kaggle
Generamos `submission.csv` con el formato que exige la competencia (`Id`, `SalePrice`).

In [ ]:
pred_test = np.expm1(mejor.predict(X_test))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": pred_test})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nPrecio medio predicho en test: ${pred_test.mean():,.0f}")

try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("(Fuera de Colab: el archivo quedó guardado como submission.csv)")

Sube `submission.csv` en la pestaña **Submit Predictions** de la competencia en Kaggle para obtener tu puntaje oficial.

## 14. Ejercicios propuestos
1. Cambia el umbral de asimetría (`0.75`) y observa cómo varía el RMSE.
2. Crea una variable nueva (por ejemplo, `OverallQual * GrLivArea`) y comprueba si mejora el modelo.
3. Promedia las predicciones de Ridge y Lasso y compara el error con cada uno por separado.
4. En la sección 4, repite la prueba t con otra variable binaria (por ejemplo, `PavedDrive`).
5. En el simulador, encuentra qué cambio aislado aumenta más el precio de una casa.